In [65]:
#Download MSWEP data from shared drive folder
#Workflow - download nc file from drive - convert to tif each - clip to roi - save to local drive

import os
import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.service_account import Credentials as ServiceAccountCredentials
from datetime import date, timedelta
import netCDF4
import rasterio
from rasterio.transform import from_origin
import numpy as np

In [ ]:
# Configuration 
CREDENTIAL_FILE = r"C:\Users\A S U S\Documents\Keys\credentials.json"
FOLDER_URL = "https://drive.google.com/drive/folders/1DVR90Ud1C444bTOPeENX-3I7tgqPLnao"
FOLDER_ID = FOLDER_URL.split('/folders/')[-1].split('?')[0]
OUTPUT_TEMP_DIR = r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MSWEP_TEMP"
OUTPUT_FIN_DIR = r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\MSWEP_TIF"
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

#variables
start_year = 1990
end_year = 2020 #exclusive

#Helper functions - intialize drive service, create file download list by taking years, convert nc to tif
def initialize_drive_service(credential_file, scopes):
    """Initializes and returns the Google Drive API service."""
    try:
        # Load credentials from service account key file
        creds = ServiceAccountCredentials.from_service_account_file(credential_file, scopes=scopes)
        print("✅ Loaded service account credentials.")
    except Exception as e:
        print(f"❌ Error loading credentials from {credential_file}: {e}")
        print("Ensure the path is correct and it is a valid Service Account JSON file.")
        return None
    
    # Build the Drive API client
    drive_service = build('drive', 'v3', credentials=creds)
    print("✅ Google Drive API client initialized successfully.")
    return drive_service

def convert_nc_to_tif(nc_path, tif_path, variable_name):
    """
    Reads a NetCDF file and saves one variable as a GeoTIFF.
    Handles 2D (lat, lon) and 3D (time, lat, lon) data.
    """
    try:
        with netCDF4.Dataset(nc_path, 'r') as nc_data:
            # --- 3. Read file from temp directory ---
            
            # Find the correct variable names for lat/lon (common variations)
            lat_var = 'latitude' if 'latitude' in nc_data.variables else 'lat'
            lon_var = 'longitude' if 'longitude' in nc_data.variables else 'lon'

            if variable_name not in nc_data.variables:
                print(f"    ❌ Variable '{variable_name}' not in {nc_path}")
                print(f"       Available: {list(nc_data.variables.keys())}")
                return False
            
            # Extract data and coordinates
            var_obj = nc_data.variables[variable_name]
            data = var_obj[:]
            lat = nc_data.variables[lat_var][:]
            lon = nc_data.variables[lon_var][:]

            # Handle 3D data (e.g., [time, lat, lon]) by taking the first slice
            if data.ndim == 3:
                print(f"    (Note: 3D data found, using first time-slice)")
                data = data[0, :, :]
            elif data.ndim != 2:
                print(f"    ❌ Error: Variable '{variable_name}' has unsupported {data.ndim} dimensions.")
                return False
            
            # Get NoData value if it exists
            nodata_val = var_obj.getncattr('_FillValue') if '_FillValue' in var_obj.ncattrs() else None

            # Determine georeferencing (assumes regular grid)
            res_lat = abs(lat[1] - lat[0])
            res_lon = abs(lon[1] - lon[0])
            # from_origin expects the *center* of the upper-left pixel
            # For MSWEP, latitude is descending, so lat.max() is correct
            transform = from_origin(lon.min(), lat.max(), res_lon, res_lat)

            # Define GeoTIFF profile
            profile = {
                'driver': 'GTiff',
                'height': data.shape[0],
                'width': data.shape[1],
                'count': 1,
                'dtype': data.dtype,
                'crs': 'EPSG:4326', # WGS84
                'transform': transform,
                'nodata': nodata_val
            }
            
            # --- 4. Save to final directory ---
            with rasterio.open(tif_path, 'w', **profile) as dst:
                dst.write(data.astype(profile['dtype']), 1) # Ensure dtype match
            
            print(f"    ✅ Converted and saved to: {tif_path}")
            return True

    except Exception as e:
        print(f"    ❌ Error during NC-to-TIF conversion: {e}")
        return False

filename_list = []
for year in range(start_year, end_year):
    start_date = date(year, 1, 1)
    if year % 4 == 0:
        end_date = date(year, 12, 31)
    else:
        end_date = date(year, 12, 31)
    for day_offset in range((end_date - start_date).days + 1):
        date_itr = start_date + timedelta(days=day_offset)
        
        for hour in range(0, 24, 3):
            year = str(date_itr.year)
            doy = (str(date_itr.timetuple().tm_yday)).zfill(3)
            filename = f'{year}{doy}.{str(hour).zfill(2)}.nc'
            filename_list.append(filename)

filename_list.sort(reverse=False)
filename_list.insert(0, '1989365.21.nc')
filename_list.insert(0, '1989365.18.nc')
print("Number of files to download:",len(filename_list))

In [ ]:
# Secondary helper functions - checking folders and files

def check_folder_exists(drive_service, folder_id):
    """Checks if the specified folder exists in Google Drive."""
    try:
        folder = drive_service.files().get(fileId=folder_id, fields='id, name').execute()
        print(f"✅ Folder found: {folder['name']} (ID: {folder['id']})")
        return True
    except Exception as e:
        print(f"❌ Error accessing folder with ID {folder_id}: {e}")
        print("Ensure the folder ID is correct and the service account has access.")
        return False
    
def list_files_in_folder(drive_service, folder_id, year_prefix):
    query = f"'{folder_id}' in parents and trashed=false"
    page_token = None
    matched_files = []
    try:
        while True:
            response = drive_service.files().list(
                q=query,
                fields="nextPageToken, files(id, name)",
                pageToken=page_token,
                pageSize=1000
            ).execute()
            for f in response.get('files', []):
                name = f.get('name', '')
                if name.startswith(str(year_prefix)):
                    matched_files.append({'id': f.get('id'), 'name': name})
            page_token = response.get('nextPageToken', None)
            if not page_token:
                break
    except Exception as e:
        print(f"❌ Error listing files: {e}")
        return []

    if not matched_files:
        print(f"No files starting with '{year_prefix}' found.")
        return []
    
    matched_files.sort(key=lambda x: x['name'])
    printed_ids = set()
    first_block = matched_files[:10]
    last_block = matched_files[-10:]

    print(f"First {len(first_block)} files starting with '{year_prefix}':")
    for item in first_block:
        print(f" - {item['name']} (ID: {item['id']})")
        printed_ids.add(item['id'])
    for item in last_block:
        print(f" - {item['name']} (ID: {item['id']})")

    return matched_files

#list_files_in_folder(drive_service, FOLDER_ID, "2000")

In [ ]:
def download_and_process_files(drive_service, folder_id, target_filenames, nc_variable_name, temp_dir, final_dir):
    """
    Main workflow function to download, convert, and save files.
    
    Args:
        drive_service: Authenticated Google Drive API service.
        folder_id (str): ID of the Drive folder to search in.
        target_filenames (list): List of exact filenames to download (e.g., ['1990001.00.nc', ...]).
        nc_variable_name (str): The variable name inside the NetCDF file to extract (e.g., 'precipitation').
        temp_dir (str): Path to local temporary directory for .nc files.
        final_dir (str): Path to local final directory for .tif files.
    """
    
    print("--- Starting Download and Processing Workflow ---")
    
    # Ensure local directories exist
    os.makedirs(temp_dir, exist_ok=True)
    os.makedirs(final_dir, exist_ok=True)
    
    total_files = len(target_filenames)
    print(f"Found {total_files} target files in the list.")
    
    # --- 1. Take file from file list one by one ---
    for i, filename in enumerate(target_filenames, 1):
        print(f"\n[{i}/{total_files}] Processing: {filename}")
        
        # Define local paths
        temp_nc_path = os.path.join(temp_dir, filename)
        final_tif_name = os.path.splitext(filename)[0] + '.tif'
        final_tif_path = os.path.join(final_dir, final_tif_name)
        
        # Skip if the final TIF file already exists
        if os.path.exists(final_tif_path):
            print(f"    SKIPPING: Final TIF already exists at {final_tif_path}")
            continue
            
        try:
            # --- 2. Match file name in drive ---
            print(f"    Searching on Google Drive...")
            query = f"name='{filename}' and '{folder_id}' in parents and trashed=false"
            response = drive_service.files().list(
                q=query,
                spaces='drive',
                fields='files(id, name)',
                pageSize=1
            ).execute()
            
            drive_files = response.get('files', [])
            
            if not drive_files:
                print(f"    ⚠️ File not found on Drive: {filename}")
                continue # Move to the next file in the list

            file_id = drive_files[0].get('id')
            print(f"    Found file on Drive (ID: {file_id})")

            # --- 3. Download matched file to temp directory ---
            print(f"    Downloading to: {temp_nc_path}")
            request = drive_service.files().get_media(fileId=file_id)
            
            with open(temp_nc_path, 'wb') as f:
                downloader = MediaIoBaseDownload(f, request)
                done = False
                while not done:
                    status, done = downloader.next_chunk()
                    if status:
                        progress = int(status.progress() * 100)
                        print(f"    Progress: {progress}%", end='\r')
            
            print(f"\n    ✅ Download complete.")

            # --- 4. Read file from temp and convert (Steps 4 & 5) ---
            print(f"    Converting to TIF...")
            success = convert_nc_to_tif(
                nc_path=temp_nc_path,
                tif_path=final_tif_path,
                variable_name=nc_variable_name
            )
            
            # --- 5. Clean up temp file ---
            if success:
                try:
                    os.remove(temp_nc_path)
                    print(f"    Removed temp file: {temp_nc_path}")
                except Exception as e:
                    print(f"    ⚠️ Warning: Could not remove temp file: {e}")
            else:
                print(f"    Conversion failed. Temp file kept for inspection: {temp_nc_path}")

        except Exception as e:
            print(f"    ❌ An unexpected error occurred for {filename}: {e}")
            
    print("\n--- Workflow Finished ---")

In [ ]:

drive_service = initialize_drive_service(CREDENTIAL_FILE, SCOPES)
download_and_process_files(
        drive_service=drive_service,
        folder_id=FOLDER_ID,
        target_filenames=filename_list,
        nc_variable_name='precipitation',
        temp_dir=OUTPUT_TEMP_DIR,
        final_dir=OUTPUT_FIN_DIR
)